# Dashboard COVID-19: ca nhiễm, tử vong và tiêm chủng

Notebook dùng compact CSV hiện hành của OWID Data Catalog để phân tích ba chỉ số trong cùng kỳ. CSV Kaggle `bolkonsky/covid19` được giữ làm nguồn dự phòng/tham chiếu vì snapshot cũ không có dữ liệu vaccine.

## 1. Cai dat thu vien

Neu chua cai dat, mo Terminal trong VS Code va chay: `pip install -r requirements.txt`.

In [1]:
import numpy as np
from pathlib import Path
import importlib
import pandas as pd
import covid_processor as cp

cp = importlib.reload(cp)
CovidDataProcessor = cp.CovidDataProcessor
DATE_WINDOW = cp.DATE_WINDOW
FIVE_CONTINENTS = cp.FIVE_CONTINENTS
SELECTED_COUNTRIES = cp.SELECTED_COUNTRIES

pd.set_option('display.max_columns', 20)

## 2. Python co ban: bon yeu cau voi du lieu COVID-19

Cac vi du nay luyen bien, ham, dieu kien va vong lap tren du lieu gia lap nho. Moi ham giai quyet mot yeu cau rieng.


In [2]:
# Yeu cau 1: phan loai muc ca moi theo nguong.
def classify_daily_cases(cases):
    """Xếp số ca mới vào nhóm Low, Medium hoặc High."""
    if cases < 1000:
        return "Low"
    elif cases < 10000:
        return "Medium"
    return "High"

# Yeu cau 2: tinh tong ca tu danh sach ban ghi bang vong lap.
def sum_reported_cases(records):
    """Cộng số ca mới trong danh sách bản ghi bằng vòng lặp."""
    total = 0
    for record in records:
        total += record["new_cases"]
    return total

# Yeu cau 3: tim quoc gia co so ca moi cao nhat bang vong lap.
def country_with_most_cases(records):
    """Trả về tên quốc gia có nhiều ca mới nhất trong danh sách."""
    highest = records[0]
    for record in records[1:]:
        if record["new_cases"] > highest["new_cases"]:
            highest = record
    return highest["country"]

# Yeu cau 4: xep nhom ty le tiem chung bang dieu kien.
def classify_coverage(rate):
    """Xếp tỷ lệ tiêm chủng vào nhóm Low, Medium, High hoặc Unknown."""
    if rate is None:
        return "Unknown"
    elif rate < 30:
        return "Low"
    elif rate <= 70:
        return "Medium"
    return "High"

sample_records = [
    {"country": "Vietnam", "new_cases": 1200, "vaccination_rate": 78.0},
    {"country": "Japan", "new_cases": 8300, "vaccination_rate": 83.0},
    {"country": "Brazil", "new_cases": 950, "vaccination_rate": 65.0},
]
print("Tong ca moi trong mau:", sum_reported_cases(sample_records))
print("Quoc gia co ca moi cao nhat:", country_with_most_cases(sample_records))
for record in sample_records:
    print(record["country"], classify_daily_cases(record["new_cases"]), classify_coverage(record["vaccination_rate"]))


Tong ca moi trong mau: 10450
Quoc gia co ca moi cao nhat: Japan
Vietnam Medium High
Japan Medium High
Brazil Low Medium


## 3. Cau truc du lieu va cau hinh

`countries=None` nghia la giu toan bo quoc gia. List quoc gia ben duoi chi dung de ve bieu do duong de de quan sat.

In [3]:
print('List - countries for line chart:', SELECTED_COUNTRIES)
print('Dict - country codes:', cp.COUNTRY_CODES)
print('Set - 5 continents:', sorted(FIVE_CONTINENTS))
print('Tuple - analysis window:', DATE_WINDOW)

previous_raw_data = globals().get('raw_data')
OUTPUT_DIR = Path('outputs')
processor = CovidDataProcessor(
    countries=None,  # None = tat ca quoc gia trong 5 chau luc
    date_window=DATE_WINDOW,
    output_dir=OUTPUT_DIR,
    visualization_countries=SELECTED_COUNTRIES,
)
if previous_raw_data is not None:
    processor.raw = previous_raw_data

List - countries for line chart: ['Vietnam', 'United States', 'India', 'Brazil', 'United Kingdom']
Dict - country codes: {'VNM': 'Viet Nam', 'USA': 'United States', 'IND': 'India', 'BRA': 'Brazil', 'GBR': 'United Kingdom'}
Set - 5 continents: ['Africa', 'America', 'Asia', 'Europe', 'Oceania']
Tuple - analysis window: ('2020-01-01', '2023-12-31')


## 4. Tải CSV OWID hiện hành (nguồn chính)

Pipeline tải compact CSV công khai từ [OWID Data Catalog](https://docs.owid.io/projects/etl/api/covid/), chứa ca nhiễm, tử vong, vaccine và mã quốc gia/ngày trong một tệp. CSV Kaggle [`bolkonsky/covid19`](https://www.kaggle.com/datasets/bolkonsky/covid19) là nguồn dự phòng; snapshot cũ trong project chỉ tới năm 2020 và thiếu vaccine.

In [4]:
raw_data = processor.load_data()
owid_countries = (
    raw_data.loc[
        raw_data['continent'].isin(cp.SOURCE_CONTINENTS)
        & ~raw_data['iso_code'].astype(str).str.startswith('OWID_'),
        ['iso_code', 'continent', 'location'],
    ]
    .drop_duplicates()
    .sort_values(['continent', 'location'])
    .reset_index(drop=True)
)
print(f'{processor.data_source}: {len(raw_data):,} rows for {owid_countries["location"].nunique()} countries and territories.')
display(owid_countries)

Loaded 617,667 rows from OWID Catalog: current compact COVID-19 CSV (cached).
OWID Catalog: current compact COVID-19 CSV (cached): 617,667 rows for 237 countries and territories.


,iso_code,continent,location
0,DZA,Africa,Algeria
1,AGO,Africa,Angola
2,BEN,Africa,Benin
3,BWA,Africa,Botswana
4,BFA,Africa,Burkina Faso
...,...,...,...
232,PRY,South America,Paraguay
233,PER,South America,Peru
234,SUR,South America,Suriname
235,URY,South America,Uruguay


### 4.1. Cào Wikipedia để đối chiếu (nguồn phụ)

Wikipedia chỉ là nguồn scraping tham khảo. Bảng này có thể cập nhật tại thời điểm khác với CSV OWID, nên dùng để đối chiếu tổng quan theo quốc gia, không ghép như chuỗi thời gian cùng kỳ.

In [5]:
import os
if os.environ.get("COVID_SKIP_WIKIPEDIA") == "1":
    wiki_summary = None
    print("Wikipedia cross-check skipped for this offline run; all analysis still uses the OWID dataset.")
else:
    wiki_summary = processor.scrape_wikipedia_summary()
    if wiki_summary is None:
        print("Wikipedia data is unavailable; the OWID dataset can still be analyzed.")
    else:
        print(f"Wikipedia loaded {len(wiki_summary):,} rows.")
        display(wiki_summary)

Scraped 239 Wikipedia rows.
Wikipedia loaded 239 rows.


,location,wiki_cases,wiki_deaths
0,World,779277425.0,7115706.0
1,Peru,4533862.0,221072.0
2,Bulgaria,1341082.0,38778.0
3,North Macedonia,352093.0,9991.0
4,Bosnia and Herzegovina,404289.0,16419.0
...,...,...,...
234,"Saint Helena, Ascension and Tristan da Cunha",2166.0,0.0
235,Pitcairn Islands,4.0,0.0
236,Tokelau,80.0,0.0
237,North Korea,0.0,0.0


## 5. Làm sạch dữ liệu theo quốc gia

Loại bỏ dòng tổng hợp không có mã quốc gia/châu lục, chuẩn hóa ngày và kiểu số. Giữ missing values là thiếu và không xóa các số điều chỉnh âm; tính các chỉ số trên một triệu dân khi nguồn chưa có giá trị tương ứng.

### 5.1. Thong ke co ban bang NumPy

Chuyen cot so ca moi trung binh 7 ngay thanh mang NumPy, sau do tinh cac thong ke mo ta. Moi dong trong du lieu tuong ung mot quoc gia-ngay, nen cac thong ke nay mo ta tap quan sat da lam sach.


In [6]:
cleaned_all = processor.clean_data()
daily_cases_array = cleaned_all["new_cases_smoothed"].dropna().to_numpy(dtype=float)
if daily_cases_array.size:
    print(f"So dong quan sat: {daily_cases_array.size:,}")
    print(f"Trung binh: {np.mean(daily_cases_array):,.2f}")
    print(f"Trung vi: {np.median(daily_cases_array):,.2f}")
    print(f"Do lech chuan: {np.std(daily_cases_array):,.2f}")
    print(f"Phan vi 90%: {np.percentile(daily_cases_array, 90):,.2f}")
else:
    print("Khong co du lieu ca moi da lam muot de tinh thong ke.")


Cleaned 343,983 country-day rows from 237 countries in 5 continents.
So dong quan sat: 337,522
Trung binh: 2,291.79
Trung vi: 15.29
Do lech chuan: 34,469.00
Phan vi 90%: 2,219.71


In [7]:
print('Continents found:', sorted(cp.CONTINENTS))
print('Countries found:', cleaned_all['location'].nunique())
cleaned_all[['continent', 'location', 'date', 'new_cases_smoothed', 'vaccination_level']].head()

Continents found: ['Africa', 'America', 'Asia', 'Europe', 'Oceania']
Countries found: 237


,continent,location,date,new_cases_smoothed,vaccination_level
0,America,Aruba,2020-01-01,NaN,Unknown
1,America,Aruba,2020-01-02,NaN,Unknown
2,America,Aruba,2020-01-03,NaN,Unknown
3,America,Aruba,2020-01-04,0.0,Unknown
4,America,Aruba,2020-01-05,0.0,Unknown


## 6. Phan tich theo quoc gia

In [8]:
country_summary_owid = processor.analyze_country_trends(include_wikipedia=False)
if wiki_summary is not None:
    country_summary_all = country_summary_owid.merge(wiki_summary, on='location', how='left')
    matched_countries = country_summary_all['wiki_cases'].notna().sum()
    print(f'Merged OWID + Wikipedia: {matched_countries}/{len(country_summary_all)} countries matched by name.')
else:
    country_summary_all = country_summary_owid
    print('Wikipedia unavailable; continuing with the OWID summary.')

processor.summary_all = country_summary_all.copy()
processor.summary = cp.filter_countries(processor.summary_all, processor.countries)
display(country_summary_all.head(15))
print('Peak dates for countries in the line chart:')
display(processor.findings['peaks'])

Merged OWID + Wikipedia: 218/237 countries matched by name.


,iso_code,location,continent,total_cases,total_deaths,cases_per_million,deaths_per_million,avg_new_cases_per_million,vaccination_rate,vaccination_level,country_code_name,wiki_cases,wiki_deaths
0,AFG,Afghanistan,Asia,230375.0,7973.0,5677.2188,196.48167,3.893840,45.270844,Medium,NaN,235214.0,7998.0
1,ALB,Albania,Europe,334596.0,3604.0,118331.5700,1274.57290,81.160198,45.244260,Medium,NaN,337234.0,3608.0
2,DZA,Algeria,Africa,272010.0,6881.0,5981.2140,151.30595,4.102341,14.251447,Low,NaN,272440.0,6881.0
3,ASM,American Samoa,Oceania,8359.0,34.0,172831.6000,702.98770,118.540187,NaN,Unknown,NaN,8359.0,34.0
4,AND,Andorra,Europe,48015.0,159.0,602280.4400,1994.43070,413.086710,67.109460,Medium,NaN,48015.0,159.0
5,AGO,Angola,Africa,106928.0,1937.0,3000.6430,54.35663,2.058054,26.965265,Low,NaN,107487.0,1937.0
6,AIA,Anguilla,America,3904.0,12.0,274890.8800,844.95140,188.539684,73.102380,High,NaN,3904.0,12.0
7,ATG,Antigua and Barbuda,America,9106.0,146.0,98072.1600,1572.42860,67.334503,67.187935,Medium,NaN,9106.0,146.0
8,ARG,Argentina,America,10044957.0,130472.0,221216.0500,2873.33230,151.725684,76.860214,High,NaN,10119840.0,130859.0
9,ARM,Armenia,Asia,451426.0,8775.0,156697.2700,3045.94430,107.474115,35.779274,Medium,NaN,454904.0,8785.0


Peak dates for countries in the line chart:


,location,date,new_cases_smoothed
43117,Brazil,2022-02-05,189526.00
140376,India,2021-05-09,391279.56
105578,United Kingdom,2022-01-06,214690.28
322830,United States,2022-01-17,809735.30
333108,Vietnam,2022-03-18,274025.28


### 6.1. Diễn giải insight

Bảng tổng hợp gồm ca nhiễm/tử vong trên một triệu dân, trung bình ca mới làm mượt và tỷ lệ tiêm đủ liều mới nhất trong kỳ. Tương quan Pearson giữa tử vong trên một triệu dân và tiêm chủng chỉ mô tả mối liên hệ giữa các quốc gia, không chứng minh quan hệ nhân quả.

In [9]:
leaders = country_summary_owid.dropna(subset=["total_cases", "total_deaths"])
if not leaders.empty:
    top_cases_row = leaders.loc[leaders["total_cases"].idxmax()]
    top_deaths_row = leaders.loc[leaders["total_deaths"].idxmax()]
    print(f"Tong ca nhiem tich luy cao nhat: {top_cases_row['location']} ({int(top_cases_row['total_cases']):,}).")
    print(f"Tong tu vong tich luy cao nhat: {top_deaths_row['location']} ({int(top_deaths_row['total_deaths']):,}).")

valid_pair = country_summary_owid[["total_cases", "total_deaths"]].dropna()
if len(valid_pair) >= 2:
    cases_deaths_correlation = np.corrcoef(valid_pair["total_cases"], valid_pair["total_deaths"])[0, 1]
    print(f"Tuong quan Pearson giua tong ca va tong tu vong theo quoc gia: {cases_deaths_correlation:.3f}.")


Tong ca nhiem tich luy cao nhat: United States (103,436,829).
Tong tu vong tich luy cao nhat: United States (1,164,499).
Tuong quan Pearson giua tong ca va tong tu vong theo quoc gia: 0.764.


## 7. Trực quan hóa

Notebook tạo ba biểu đồ theo yêu cầu: đường ca mới trung bình 7 ngày trên một triệu dân theo thời gian; heatmap Pearson gồm ca nhiễm, tử vong, ca mới và tiêm chủng; histogram tỷ lệ hoàn thành phác đồ vaccine ban đầu theo quốc gia. Dashboard hiển thị thêm xu hướng ca mới, tử vong và tỷ lệ tiêm chủng theo tháng.

In [10]:
chart_files = processor.plot_all(show=False)
print('Saved charts:')
for chart in chart_files:
    print('-', chart)

Saved charts:
- outputs\chart1_line_new_cases.png
- outputs\chart2_heatmap_covid_indicators.png
- outputs\chart3_hist_vaccination_rate.png


## 8. Xuat du lieu va bao cao mini

Cell tiep theo xuat CSV da lam sach, cac bang tong hop, JSON cho dashboard, bao cao mini va audit chat luong du lieu. Sau khi chay notebook, mo `dashboard.html` bang Live Server de loc chau luc, tim quoc gia va xem cac bieu do tuong tac.

In [11]:
cleaned_file, summary_file, dashboard_file, report_file = processor.export_results()
print('Created:')
audit_file = OUTPUT_DIR / 'data_quality_audit.csv'
for file in [cleaned_file, summary_file, dashboard_file, report_file, audit_file]:
    print('-', file)
print('\n' + report_file.read_text(encoding='utf-8'))

Created:
- outputs\cleaned_covid_all_countries.csv
- outputs\country_summary_all.csv
- outputs\dashboard_data.json
- outputs\summary_report.md
- outputs\data_quality_audit.csv

# Báo cáo mini: Phân tích dữ liệu COVID-19

## Mục tiêu và nguồn dữ liệu

Nguồn chính của lần chạy này là compact CSV hiện hành từ OWID Data Catalog (https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv; tài liệu: https://docs.owid.io/projects/etl/api/covid/). Kaggle `bolkonsky/covid19` vẫn là nguồn dự phòng và tài liệu tham chiếu; snapshot cũ của Kaggle trong project chỉ tới 19/09/2020 nên không đủ để phân tích vaccine. Notebook cũng cào bảng tổng hợp từ Wikipedia để đối chiếu tham khảo; Đã ghép dữ liệu Wikipedia theo tên cho 218 quốc gia.

## Phương pháp xử lý và phân tích

Pipeline đọc compact CSV OWID đã cache hoặc tải trực tiếp; nếu không truy cập được catalog thì dùng CSV Kaggle làm dự phòng. Các dòng tổng hợp và nhóm khu vực không có mã quốc gia được loại bỏ; khoảng ngày yêu cầu là *